# 04 — Toàn bộ text trên Kaggle, kế hoạch GPU riêng: account 0

Add Input **ALL_TEXT_DATASET_READY** và **GPU_PLAN_DATASET_READY**.
Các tài khoản dùng cùng hai dataset; account ID đã đặt sẵn 0.
Bản này yêu cầu kế hoạch 2 acc, mỗi acc T4 x2; gỡ kế hoạch cũ trước khi chạy.
Chủ dataset cấp quyền đọc cả hai gói cho các tài khoản còn lại.

Có thể chọn CPU và chỉ chạy cell 1 để kiểm đủ file thư viện và kế hoạch.
Cell này không nạp weights hoặc mở Drive. ALL_TEXT_INPUTS_PRESENT xác nhận
toàn bộ text task đã đóng gói hiện diện; số task của kế hoạch có thể ít hơn.

Chạy GPU: T4 x2, Internet ON, Save Version / Save & Run All. Mỗi GPU có
journal theo đúng library+plan; task xong được giữ, text hash trùng được dùng
lại. Runtime/quota hoặc deadline có thể khiến kế hoạch còn dở. Outputs phải
lưu thành công trước khi nhập về Drive. Sau khi nhập cả hai tài khoản,
Colab tạo kế hoạch nhỏ mới; giữ dataset text và thay dataset kế hoạch.

Log đầy đủ nằm ở vmb_full_queue/gpu-worker.log trong Outputs. Màn hình
chỉ hiện tiến độ và GPU_RUN_SUMMARY để tránh IOPub timeout vì JSON quá lớn.
Không sửa worker khóa trong dataset, không xuất lại thư viện text.
Nếu process_state=FAILED nhưng có partial manifest hợp lệ, notebook vẫn
kết thúc để giữ Outputs; đó chưa phải hoàn tất embedding.

Giữ nguyên text đã freeze, gồm cả cờ LOW. Kiểm kê file không sửa chất lượng
ngữ nghĩa. Đây là corpus child embedding, chưa phải query/index/submission.

In [ ]:
#@title 1. Kiểm tra toàn bộ thư viện text + kế hoạch (CPU được, chưa nạp model)
import hashlib
import importlib
import json
import os
from pathlib import Path
import re
import subprocess
import sys
import time
SESSION_STARTED=time.monotonic()
KAGGLE_ACCOUNT_ID=0
EXPECTED_EXECUTION_ACCOUNTS=2
SESSION_HOURS=11.5
print('ALL-TEXT Kaggle preflight | account',KAGGLE_ACCOUNT_ID,flush=True)
if not Path('/kaggle/input').is_dir() or Path('/var/colab/hostname').is_file():
    raise RuntimeError('GPU notebook chạy trên Kaggle; Colab dùng notebook xuất full-data.')
libraries,plans={},{}
for path in Path('/kaggle/input').rglob('job.json'):
    item=json.loads(path.read_text())
    if item.get('schema')=='full-data-text-library-v1':
        libraries[item.get('manifest_sha256')]=(path.parent,item)
    elif item.get('schema') in ('full-data-execution-plan-v1','full-data-execution-plan-v2'):
        plans[item.get('manifest_sha256')]=(path,item)
if len(libraries)!=1:
    raise RuntimeError('Add Input đúng MỘT dataset ALL_TEXT_DATASET_READY; gỡ gói 1.84 GB/700k cũ khỏi notebook này.')
JOB_DIR,JOB=next(iter(libraries.values()))
compatible=[(path,plan) for path,plan in plans.values() if plan.get('library_job')==JOB['manifest_sha256']]
if len(compatible)!=1:
    raise RuntimeError('Add Input đúng MỘT GPU_PLAN_DATASET_READY khớp thư viện text; gỡ kế hoạch cũ khi đổi lượt.')
PLAN_PATH,PLAN=compatible[0]
if PLAN.get('team')!={'accounts':EXPECTED_EXECUTION_ACCOUNTS,'gpu_count':2}:
    raise ValueError('Dùng kế hoạch GPU mới đúng số acc đã cấu hình; gỡ kế hoạch 2 acc cũ trước khi chạy GPU.')
if JOB.get('repo_url')!='https://github.com/Platypus27-coder/VietMedBridge.git' or not re.fullmatch('[0-9a-f]{40}',JOB.get('code_commit','')):
    raise ValueError('Library producer repo/commit invalid.')
if not 0<=KAGGLE_ACCOUNT_ID<PLAN['team']['accounts'] or PLAN['team']['gpu_count']!=2:
    raise ValueError('Account ID/GPU team differs from exported library.')
expected_sources={'new_data_kaggle.py','new_data_embeddings.py','new_data_cache_lookup.py',
    'new_data_drive_api.py','full_queue_embeddings.py'}
if set(JOB['worker_sources'])!=expected_sources:
    raise ValueError('Missing locked library/worker source.')
EXECUTOR_DIR=PLAN_PATH.parent if PLAN['schema']=='full-data-execution-plan-v2' else JOB_DIR
if PLAN['schema']=='full-data-execution-plan-v2':
    expected_executors={'full_queue_embeddings.py','new_data_kaggle.py'}
    if PLAN.get('qwen_execution') in ('qwen-token-batches-v1','qwen-token-batches-v2'):
        expected_executors.add('qwen_token_batching.py')
    if set(PLAN.get('executor_sources',{}))!=expected_executors:
        raise ValueError('Missing locked execution scheduler sources.')
    for name,checksum in PLAN['executor_sources'].items():
        if hashlib.sha256((EXECUTOR_DIR/name).read_bytes()).hexdigest()!=checksum:
            raise ValueError('Execution scheduler checksum changed.')
if PLAN.get('qwen_execution') not in ('qwen-token-batches-v1','qwen-token-batches-v2'):
    SESSION_HOURS=min(SESSION_HOURS,10.0)
    print('LEGACY_EXECUTION_PLAN: giữ deadline cũ; bản tối ưu cần kế hoạch mới sau khi nhập đủ Outputs.',flush=True)
else:
    print('QWEN_TOKEN_BATCHING_READY: benchmark trên GPU; deadline 11.5 giờ tính từ cell đầu.',flush=True)
"""Inspect an uploaded round using local Kaggle metadata before loading weights.

This proves task-file presence and job accounting, not semantic text quality.
Native workers still check each payload SHA and text key when consuming it.
The source is embedded into notebooks so existing uploaded jobs stay immutable.
"""
from collections import Counter
import hashlib
import json
from pathlib import Path
import re


def inspect_job_inputs(directory, job, *, expected_scope=None, report=print):
    directory = Path(directory)
    digest = hashlib.sha256(json.dumps(
        {k: v for k, v in job.items() if k != 'manifest_sha256'},
        sort_keys=True, ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()
    if job.get('manifest_sha256') != digest:
        raise ValueError('Job header changed; do not load model weights.')
    team = job.get('team', {})
    accounts, gpus = team.get('accounts'), team.get('gpu_count')
    if type(accounts) is not int or accounts < 1 or type(gpus) is not int or gpus < 1:
        raise ValueError('Invalid account/GPU assignment.')
    if expected_scope is not None:
        scope = job.get('coverage_scope', {})
        sources = job.get('frozen_sources', [])
        if (scope.get('scope_manifest_sha256') != expected_scope
                or type(scope.get('official_ids')) is not int or scope['official_ids'] < 802028
                or scope.get('source_builds') != 13 or len(sources) != 13
                or len({s.get('build_run') for s in sources}) != 13):
            raise ValueError('This is not the verified 802,028-ID / 13-source full-data scope.')
    tasks = job.get('tasks', [])
    if not tasks or job.get('scheduled_tasks') != len(tasks):
        raise ValueError('Scheduled task count differs from the exported round.')
    remaining = job.get('remaining_tasks_before_round', {})
    queued = job.get('unscheduled_tasks')
    if (type(queued) is not int or queued < 0
            or any(type(v) is not int or v < 0 for v in remaining.values())
            or sum(remaining.values()) != len(tasks) + queued
            or job.get('all_remaining_tasks_scheduled') is not (queued == 0)):
        raise ValueError('Pending task accounting changed; no GPU work started.')
    files = {p.name for p in directory.iterdir() if p.is_file()}
    seen, missing = set(), []
    by_model, by_account = {}, {}
    actual_tasks = Counter()
    for task in tasks:
        task_id, family, worker, rows = (task.get(k) for k in ('id', 'family', 'worker', 'rows'))
        if (not isinstance(task_id, str) or not re.fullmatch('[0-9a-f]{64}', task_id)
                or task_id in seen or task.get('selection_id') != task_id
                or family not in job.get('models', {})
                or type(worker) is not int or not 0 <= worker < accounts*gpus
                or type(rows) is not int or not 1 <= rows <= 512
                or task.get('asset') != 'task-'+task_id+'.parquet'
                or not re.fullmatch('[0-9a-f]{64}', task.get('sha256', ''))):
            raise ValueError('Invalid or duplicate task in uploaded round.')
        seen.add(task_id)
        if task['asset'] not in files:
            missing.append(task['asset'])
        actual_tasks[family] += 1
        for groups, key in ((by_model, family), (by_account, worker//gpus)):
            counts = groups.setdefault(key, {'tasks': 0, 'text_rows': 0})
            counts['tasks'] += 1
            counts['text_rows'] += rows
    if any(actual_tasks[k] > remaining.get(k, -1) for k in actual_tasks):
        raise ValueError('Model task count exceeds the retained queue.')
    if missing:
        raise FileNotFoundError(f'Uploaded dataset is missing {len(missing)} task files; '
                                f'no model weights loaded. Examples: {missing[:3]}')
    for name, checksum in job.get('worker_sources', {}).items():
        if name not in files or Path(name).name != name:
            raise FileNotFoundError('Uploaded worker source is missing: '+name)
        if hashlib.sha256((directory/name).read_bytes()).hexdigest() != checksum:
            raise ValueError('Uploaded worker source changed: '+name)
    summary = {'state': 'INPUT_INVENTORY_COMPLETE', 'job': job['manifest_sha256'],
               'scope': job.get('coverage_scope', {}), 'team': team, 'tasks': len(tasks),
               'queued_tasks_on_drive': queued, 'by_model': by_model, 'by_account': by_account,
               'payload_verification': 'SHA256_AND_TEXT_KEYS_ON_USE',
               'semantic_quality': 'NOT_CERTIFIED_BY_FILE_INVENTORY'}
    report('INPUT_INVENTORY_COMPLETE: all '+str(len(tasks))+' task files present; no model weights loaded.')
    report(json.dumps(summary, ensure_ascii=False, indent=2))
    return summary


def inspect_full_queue_inputs(directory, library, plan, *, expected_scope, report=print):
    """Inventory the complete text library separately from the bounded GPU plan."""
    def checked(value):
        checksum = hashlib.sha256(json.dumps(
            {k: v for k, v in value.items() if k != 'manifest_sha256'},
            sort_keys=True, ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()
        if value.get('manifest_sha256') != checksum:
            raise ValueError('Full-input library or execution plan header changed.')
        return value
    checked(library); checked(plan)
    elastic = plan.get('schema') == 'full-data-execution-plan-v2'
    if library.get('schema') != 'full-data-text-library-v1' or plan.get('schema') not in ('full-data-execution-plan-v1', 'full-data-execution-plan-v2'):
        raise ValueError('Add one all-text library and one matching execution-plan dataset.')
    scope = library.get('coverage_scope', {})
    sources = library.get('frozen_sources', [])
    if (scope.get('scope_manifest_sha256') != expected_scope or scope.get('official_ids', 0) < 802028
            or scope.get('source_builds') != 13 or len(sources) != 13
            or len({s.get('build_run') for s in sources}) != 13
            or library.get('input_export_complete') is not True):
        raise ValueError('The full-input library does not cover the verified 13-source scope.')
    if (plan.get('library_job') != library['manifest_sha256'] or (not elastic and plan.get('team') != library.get('team'))
            or (elastic and plan.get('library_team') != library.get('team'))):
        raise ValueError('Execution plan belongs to another text library or GPU team.')
    directory = Path(directory)
    files = {p.name for p in directory.iterdir() if p.is_file()}
    by_model, tasks, missing = {}, {}, set()
    for task in library.get('tasks', []):
        task_id = task.get('id')
        asset = task.get('asset', '')
        if (not isinstance(task_id, str) or not re.fullmatch('[0-9a-f]{64}', task_id)
                or task_id in tasks or task.get('selection_id') != task_id
                or not asset or Path(asset).name != asset or '\\' in asset
                or task.get('family') not in library.get('models', {})
                or type(task.get('rows')) is not int or not 1 <= task['rows'] <= 512
                or type(task.get('offset')) is not int or task['offset'] < 0):
            raise ValueError('Invalid task in all-input library.')
        tasks[task_id] = task
        if asset not in files:
            missing.add(asset)
        counts = by_model.setdefault(task['family'], {'tasks': 0, 'text_rows': 0})
        counts['tasks'] += 1; counts['text_rows'] += task['rows']
    if not tasks or library.get('unscheduled_tasks') != 0 or library.get('scheduled_tasks') != len(tasks):
        raise ValueError('All-input library task accounting is incomplete.')
    if missing:
        raise FileNotFoundError(f'Full-input dataset is missing {len(missing)} shared text files: {sorted(missing)[:3]}')
    planned = plan.get('tasks', [])
    if len(set(planned)) != len(planned) or not set(planned) <= set(tasks):
        raise ValueError('Execution plan contains duplicate or foreign tasks.')
    team = plan['team']
    if (type(team.get('accounts')) is not int or team['accounts'] < 1
            or type(team.get('gpu_count')) is not int or team['gpu_count'] != 2):
        raise ValueError('Invalid execution account/GPU team.')
    if elastic and (set(plan.get('task_workers', {})) != set(planned)
                    or any(type(w) is not int or not 0 <= w < team['accounts']*team['gpu_count']
                           for w in plan['task_workers'].values())):
        raise ValueError('Elastic execution task/GPU ownership changed.')
    for name, checksum in library.get('worker_sources', {}).items():
        if name not in files or Path(name).name != name:
            raise FileNotFoundError('Uploaded worker source is missing: '+name)
        if hashlib.sha256((directory/name).read_bytes()).hexdigest() != checksum:
            raise ValueError('Uploaded worker source changed: '+name)
    result = {'state': 'ALL_TEXT_INPUTS_PRESENT', 'library_job': library['manifest_sha256'],
              'scope': scope, 'all_exported_tasks': len(tasks), 'shared_text_files': len({t['asset'] for t in tasks.values()}),
              'by_model': by_model, 'this_gpu_plan_tasks': len(planned), 'execution_team': team,
              'input_export_complete': True, 'payload_verification': 'SHA256_AND_TEXT_KEYS_ON_USE',
              'semantic_quality': 'NOT_CERTIFIED_BY_FILE_INVENTORY'}
    report('ALL_TEXT_INPUTS_PRESENT: entire library present; GPU execution is planned separately.')
    report(json.dumps(result, ensure_ascii=False, indent=2))
    return result

PREFLIGHT=inspect_full_queue_inputs(JOB_DIR,JOB,PLAN,expected_scope='0310f6d51a13bdff63cb9d9dfe337d9aee8898df9f34857c84a2708d9f941721')


In [ ]:
# GPU setup: run only after reviewing the input inventory.
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=True)
CHECKOUT = Path('/kaggle/temp/VietMedBridge-new-data')
WORK_DIR = Path('/kaggle/temp/vmb_new_data')
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(WORK_DIR/'hf_cache')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
if not (CHECKOUT/'.git').is_dir():
    subprocess.run(['git','clone','--no-checkout',JOB['repo_url'],str(CHECKOUT)], check=True)
if subprocess.check_output(['git','-C',str(CHECKOUT),'remote','get-url','origin'], text=True).strip() != JOB['repo_url']:
    raise ValueError('Checkout belongs to another repository.')
subprocess.run(['git','-C',str(CHECKOUT),'fetch','--depth','1','origin',JOB['code_commit']], check=True)
subprocess.run(['git','-C',str(CHECKOUT),'checkout','--detach','FETCH_HEAD'], check=True)
print('Installing pinned encoder dependencies; no Google Drive mount on Kaggle.', flush=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.[strong]'], cwd=CHECKOUT, check=True)
sys.path[:0] = [str(JOB_DIR),str(CHECKOUT/'src')]
importlib.invalidate_caches()
from vietmedbridge.portable_embeddings import runtime_install_commands
for command in runtime_install_commands(JOB['runtime']):
    subprocess.run(command, check=True)
env = os.environ.copy()
env['PYTHONPATH'] = str(CHECKOUT/'src')+os.pathsep+str(JOB_DIR)
probe = "import torch; print('FRESH CUDA:',torch.__version__,torch.cuda.is_available(),torch.cuda.device_count(),flush=True); assert torch.cuda.is_available() and torch.cuda.device_count()==2"
subprocess.run([sys.executable,'-u','-c',probe], env=env, check=True)
print('READY | batch',JOB['batch'],'| team',JOB['team'],'| account',KAGGLE_ACCOUNT_ID,flush=True)
env['PYTHONPATH']=os.pathsep.join((str(CHECKOUT/'src'),str(EXECUTOR_DIR),str(JOB_DIR)))
print('EXECUTION_TEAM:',PLAN['team'],'| account',KAGGLE_ACCOUNT_ID,flush=True)

In [ ]:
#@title 3. Chạy phần được giao; log gọn và lưu checkpoint vào Outputs
"""Keep verbose locked worker output off Jupyter's IOPub channel.

The original dataset worker, plan and vector files stay unchanged. Its complete
stdout/stderr is saved in Kaggle Outputs; the notebook receives bounded progress
messages and a small result summary instead of millions of text-hash lines.
"""
import hashlib
import json
from pathlib import Path
import subprocess
import time


_PROGRESS_PREFIXES = (
    'GPU worker ', 'KAGGLE_CHECKPOINT ', 'Qwen corpus execution:',
    'BAAI/bge-m3 | saved mini-block ',
    'Qwen/Qwen3-Embedding-8B | saved mini-block ',
)


def _read_progress(stream, position):
    stream.seek(0, 2)
    end = stream.tell()
    if end <= position:
        return end, []
    # A finishing worker can print a very large JSON manifest in one burst.
    # The complete bytes remain in the file; inspect only a bounded tail here.
    start = max(position, end - 65536)
    stream.seek(start)
    lines = stream.read(end - start).decode('utf-8', errors='replace').splitlines()
    if start > position and lines:
        lines = lines[1:]
    progress = [line[:350] for line in lines
                if line.startswith(_PROGRESS_PREFIXES)]
    return end, progress[-4:]


def _result_summary(path, *, job, plan, account):
    value = json.loads(Path(path).read_text(encoding='utf-8'))
    supplied = value.get('manifest_sha256')
    content = {name: item for name, item in value.items() if name != 'manifest_sha256'}
    checksum = hashlib.sha256(json.dumps(content, sort_keys=True, ensure_ascii=False,
                                        separators=(',', ':')).encode('utf-8')).hexdigest()
    if (supplied != checksum or value.get('schema') != 'new-data-kaggle-result-v1'
            or value.get('job') != job or value.get('execution_plan') != plan
            or value.get('account_id') != account
            or value.get('state') not in ('ROUND_COMPLETE', 'CHECKPOINTED_PARTIAL')
            or not isinstance(value.get('tasks'), list)
            or not isinstance(value.get('blocks'), list)):
        raise ValueError('Saved result belongs to another job/plan/account or changed; retain its Outputs.')
    return {'state': value['state'], 'account_id': account,
            'completed_tasks': len(value['tasks']), 'vector_blocks': len(value['blocks']),
            'job': job, 'execution_plan': plan, 'manifest_sha256': supplied}


def run_with_bounded_output(command, *, env, log_path, result_path, job, plan,
                            account, report=print, poll_seconds=2,
                            heartbeat_seconds=30):
    """Run the immutable worker and retain partial artifacts even on failure.

    A valid partial manifest permits a normal notebook finish so Kaggle can save
    its files. A failed worker is explicitly reported as FAILED, never COMPLETE.
    CPU import still verifies every task, producer and vector before reuse.
    """
    log_path, result_path = Path(log_path), Path(result_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    last_heartbeat, last_progress = started, 'starting locked GPU worker'
    with log_path.open('ab', buffering=0) as output:
        output.write(b'\n--- KAGGLE GPU RUN ---\n')
        with log_path.open('rb') as reader:
            position = log_path.stat().st_size
            process = subprocess.Popen(command, env=env, stdout=output,
                                       stderr=subprocess.STDOUT)
            try:
                while True:
                    position, messages = _read_progress(reader, position)
                    for message in messages:
                        report(message)
                        last_progress = message
                    now = time.monotonic()
                    if now - last_heartbeat >= heartbeat_seconds:
                        report(f'KAGGLE heartbeat | {now-started:.0f}s | {last_progress}'
                               f' | full log: {log_path.name}')
                        last_heartbeat = now
                    if process.poll() is not None:
                        # Capture progress flushed between the read and exit.
                        _, final_messages = _read_progress(reader, position)
                        for message in final_messages:
                            report(message)
                        break
                    time.sleep(poll_seconds)
            finally:
                if process.poll() is None:
                    process.terminate()
                    try:
                        process.wait(timeout=30)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait(timeout=10)
    returncode = process.returncode
    if returncode:
        report(f'GPU_RUN_FAILED | return code {returncode} | full traceback: {log_path}')
    if not result_path.is_file():
        raise RuntimeError(f'GPU worker exited {returncode} without a result manifest. '
                           f'Retain Outputs and inspect {log_path}; do not restart the old plan blindly.')
    summary = _result_summary(result_path, job=job, plan=plan, account=account)
    summary.update(process_state='COMPLETE' if returncode == 0 else 'FAILED',
                   returncode=returncode, log_path=str(log_path), result_path=str(result_path))
    report('GPU_RUN_SUMMARY: ' + json.dumps(summary, ensure_ascii=False))
    if returncode:
        report('PARTIAL_OUTPUTS_READY_FOR_SAVE: worker failed; completed tasks remain available '
               'for verified CPU import. This is not full embedding completion.')
    return summary

elapsed=(time.monotonic()-SESSION_STARTED)/3600
if elapsed>=SESSION_HOURS-0.1:
    raise RuntimeError('Start a fresh saved version; bootstrap used the save deadline.')
command=[sys.executable,'-u',str(EXECUTOR_DIR/'full_queue_embeddings.py'),'launch',
    '--job',str(JOB_DIR),'--plan',str(PLAN_PATH),'--checkout',str(CHECKOUT),
    '--output','/kaggle/working/vmb_full_queue','--work',str(WORK_DIR/'workers'),
    '--account-id',str(KAGGLE_ACCOUNT_ID),'--hours',str(SESSION_HOURS-elapsed)]
GPU_RESULT=run_with_bounded_output(command,env=env,
    log_path=Path('/kaggle/working/vmb_full_queue/gpu-worker.log'),
    result_path=Path('/kaggle/working/vmb_full_queue/result/new-data-result.json'),
    job=JOB['manifest_sha256'],plan=PLAN['manifest_sha256'],account=KAGGLE_ACCOUNT_ID,
    report=lambda message:print(message,flush=True))
print('Chờ Save & Run All lưu Outputs hoàn tất, rồi nhập về Drive bằng notebook full-data CPU.',flush=True)